# Qwen2.5-1.5B × opc-sft-stage2 leaderboard — full-polar placement (r=256, 9000 steps)

Third-model **placement** on the full-polar (ns=8) ↔ partial-polar (ns=5) axis: is OLMo's full-polar lr-aversion an outlier or common? Qwen2.5-1.5B (lineage distinct from OLMo/Llama) × `opc-sft-stage2` × r=256 × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × H200. `max_steps=9000`, `eval_every=250`.

- **AdamW** (speed target): η ∈ {1e-5, 3e-5, 1e-4, 3e-4, 1e-3}
- **chord-tight k=1, ns ∈ {5, 8}**: η ∈ {3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1}

Labels/colors from the shared `canonical_label` / `canonical_colors` (AdamW black + first; every axis explicit; the figure's `assert_label_discriminates` guard hard-errors on any silent merge). Read ns=8-vs-ns=5 around Qwen's own optimum (±1 grid step). σ proxy 0.0017. Caveat: Qwen2.5 is 1.5B vs the 1B OLMo/Llama (size confound; placement is about curve shape).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure, canonical_label
from IPython.display import display

def ns_of(cfg):
    oc = cfg.get('optimizer_config') or {}
    return cfg.get('muon_ns_steps', oc.get('ns_steps'))
def picard_of(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def render(groups, suptitle):
    runs = load_runs(where={'log_group': groups}, logs_root='../logs',
                     warn_cross_commit=False, quiet=True)
    dedup = {}
    for cfg, hist in runs:
        key = (cfg['optimizer'], float(cfg['lr']), ns_of(cfg), picard_of(cfg))
        if key not in dedup or len(hist) > len(dedup[key][1]):
            dedup[key] = (cfg, hist)
    labeled = [(c, h) for c, h in dedup.values() if canonical_label(c) is not None]
    labels = {canonical_label(c) for c, _ in labeled}
    print(f'{len(labeled)} labeled runs -> {sorted(labels)}')
    fig, tdf, sdf = compare_variants_figure(
        variants={l: {} for l in labels}, common_where={}, ref_label='AdamW',
        target_label='AdamW', sigma_ref=0.0017, suptitle=suptitle,
        max_steps=9000, allow_partial=True,
        prefetched_runs=labeled, variant_key=canonical_label)
    display(tdf.style.format('{:.4f}', na_rep='—'))
    display(sdf.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                              'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'}, na_rep='—'))
    plt.show()
    return tdf, sdf

## r=256 — placement: full polar (ns=8) vs partial (ns=5) vs AdamW

`allow_partial=True`; re-run as the sweep fills in (full results ~4 h).

In [ ]:
GROUPS = [
    'qwen25_opc_r256_adamw_gpuxl',   # AdamW baseline (defines the speed target)
    'qwen25_opc_r256_chord_gpuxl',   # chord-tight k=1, ns in {5, 8}
]
tdf, sdf = render(GROUPS, 'Qwen2.5-1.5B × opc-sft-stage2 × r=256 — full polar (ns=8) vs partial (ns=5) vs AdamW')